# Combine GEE Features and PM25 Data - V2 (Tolerance-Based Matching)

This notebook merges two CSV files with **tolerance-based coordinate matching** to account for floating-point precision differences:
- **gee_features_daily.csv**: Environmental features (temperature, pressure, wind, NO2, CO, O3, AOD)
- **pm25.csv**: PM2.5 air quality measurements and sensor locations

## Key Differences from V1:
- Coordinates are rounded to 5 decimal places (~1.1 meter precision)
- Allows small floating-point variations to be matched as the same location
- Original coordinate values are preserved in output
- Reduces false mismatches due to measurement/processing errors

The merge is performed on three key fields:
- `date`: Calendar date
- `latitude`: Geographic latitude (rounded to 5 decimals for matching)
- `longitude`: Geographic longitude (rounded to 5 decimals for matching)

## Step 0: Import Libraries and Setup Paths

In [7]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Get the base directory (workspace root)
notebook_dir = Path.cwd()
# If running from notebooks folder, go up one level
if notebook_dir.name == 'notebooks':
    base_dir = notebook_dir.parent
else:
    base_dir = notebook_dir

data_dir = base_dir / 'data'

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"\nData files present:")
for file in data_dir.glob('*.csv'):
    print(f"  - {file.name}")

Base directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality
Data directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\data

Data files present:
  - combined_gee_pm25.csv
  - gee_features_daily.csv
  - pm25.csv


## Step 1: Load Data Files

In [8]:
print("Loading data files...\n")

# Load the CSV files
pm25_df = pd.read_csv(data_dir / 'pm25.csv')
gee_df = pd.read_csv(data_dir / 'gee_features_daily.csv')

print(f"PM25 data loaded: {pm25_df.shape[0]} rows × {pm25_df.shape[1]} columns")
print(f"GEE data loaded: {gee_df.shape[0]} rows × {gee_df.shape[1]} columns")

print(f"\nPM25 columns: {list(pm25_df.columns)}")
print(f"\nGEE columns: {list(gee_df.columns)}")

Loading data files...

PM25 data loaded: 88041 rows × 5 columns
GEE data loaded: 164208 rows × 13 columns

PM25 columns: ['date', 'latitude', 'longitude', 'pm25', 'sensor_name']

GEE columns: ['date', 'datetime_utc', 'id', 'lat', 'lon', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']


## Step 2: Preview Data

In [9]:
print("First few rows of PM25 data:")
print(pm25_df.head())

print("\nFirst few rows of GEE data:")
print(gee_df.head())

First few rows of PM25 data:
         date   latitude  longitude  pm25              sensor_name
0  2016-11-21  38.788056  -0.697222   4.0                ONTINYENT
1  2016-11-21  39.456111  -0.375833   3.0  VALÈNCIA-PISTA DE SILLA
2  2016-11-21  39.480300  -0.336400   2.0      VALÈNCIA-POLITÈCNIC
3  2016-11-21  39.481100  -0.408300   5.0  VALÈNCIA - MOLÍ DEL SOL
4  2016-11-21  39.481111  -0.447222   2.0          QUART DE POBLET

First few rows of GEE data:
         date         datetime_utc  id        lat       lon  \
0  2016-11-21  2016-11-21 00:00:00  90  39.235833  9.115000   
1  2016-11-21  2016-11-21 00:00:00  91  39.260556  9.136389   
2  2016-11-22  2016-11-22 00:00:00  28  36.203880 -5.383710   
3  2016-11-22  2016-11-22 00:00:00  40  41.426100  2.148000   
4  2016-11-22  2016-11-22 00:00:00  51  41.424066  2.185757   

   temperature_celsius  pressure_mb    wind_u    wind_v  NO2  CO  O3    AOD  
0            15.842538   994.344313 -2.613506  2.663921  NaN NaN NaN  0.207  
1    

## Step 3: Standardize Column Names

In [10]:
print("Standardizing column names...")
print(f"Original GEE columns: {list(gee_df.columns)}")

# Rename lat/lon to latitude/longitude in GEE data for consistency
gee_df = gee_df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})

print(f"After rename: {list(gee_df.columns)}")
print("✓ Renamed: lat → latitude, lon → longitude")

Standardizing column names...
Original GEE columns: ['date', 'datetime_utc', 'id', 'lat', 'lon', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']
After rename: ['date', 'datetime_utc', 'id', 'latitude', 'longitude', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD']
✓ Renamed: lat → latitude, lon → longitude


## Step 4: Convert Date Columns to Datetime Format

In [11]:
print("Converting date columns to datetime format...\n")

pm25_df['date'] = pd.to_datetime(pm25_df['date'], format='%Y-%m-%d')
gee_df['date'] = pd.to_datetime(gee_df['date'], format='%Y-%m-%d')

print(f"PM25 date range: {pm25_df['date'].min().date()} to {pm25_df['date'].max().date()}")
print(f"GEE date range: {gee_df['date'].min().date()} to {gee_df['date'].max().date()}")

print(f"\nPM25: {pm25_df['date'].nunique()} unique dates")
print(f"GEE: {gee_df['date'].nunique()} unique dates")

Converting date columns to datetime format...

PM25 date range: 2016-11-21 to 2026-04-16
GEE date range: 2016-11-21 to 2026-04-06

PM25: 3131 unique dates
GEE: 2393 unique dates


## Step 5: Configure Tolerance and Create Rounded Coordinates

**Tolerance explanation:**
- 5 decimal places = ~1.1 meter precision at the equator
- This handles floating-point rounding differences from sensors/data processing
- Coordinates with small variations (< 1.1m) are treated as the same location

In [12]:
# Define tolerance level (decimal places for rounding)
# 5 decimal places ≈ 1.1 meters precision
TOLERANCE_DECIMALS = 5

print(f"Coordinate Matching Tolerance:")
print(f"  Decimal places: {TOLERANCE_DECIMALS}")
print(f"  Precision: ~1.1 meters (at equator)")
print(f"  Tolerance range: ±0.000005 degrees (~0.56 meters)\n")

# Create rounded coordinate columns for matching
print("Creating rounded coordinate columns for tolerance-based matching...")

# Make copies to preserve original coordinates
pm25_match = pm25_df.copy()
gee_match = gee_df.copy()

# Add rounded coordinates for matching
pm25_match['lat_rounded'] = pm25_match['latitude'].round(TOLERANCE_DECIMALS)
pm25_match['lon_rounded'] = pm25_match['longitude'].round(TOLERANCE_DECIMALS)

gee_match['lat_rounded'] = gee_match['latitude'].round(TOLERANCE_DECIMALS)
gee_match['lon_rounded'] = gee_match['longitude'].round(TOLERANCE_DECIMALS)

print("✓ Rounded coordinates created")

# Show examples of rounding
print(f"\nExample coordinate rounding:")
sample_coords = pm25_df[['latitude', 'longitude']].drop_duplicates().head(3)
for idx, row in sample_coords.iterrows():
    rounded_lat = round(row['latitude'], TOLERANCE_DECIMALS)
    rounded_lon = round(row['longitude'], TOLERANCE_DECIMALS)
    print(f"  Original:  ({row['latitude']}, {row['longitude']})")
    print(f"  Rounded:   ({rounded_lat}, {rounded_lon})")
    print()

Coordinate Matching Tolerance:
  Decimal places: 5
  Precision: ~1.1 meters (at equator)
  Tolerance range: ±0.000005 degrees (~0.56 meters)

Creating rounded coordinate columns for tolerance-based matching...
✓ Rounded coordinates created

Example coordinate rounding:
  Original:  (38.78805555034829, -0.6972222199986008)
  Rounded:   (38.78806, -0.69722)

  Original:  (39.456111109893726, -0.3758333298316526)
  Rounded:   (39.45611, -0.37583)

  Original:  (39.4802999999235, -0.3363999999743331)
  Rounded:   (39.4803, -0.3364)



## Step 6: Merge Datasets on Date and Rounded Coordinates

#### Merge Strategy:
1. Join on `(date, lat_rounded, lon_rounded)` to allow tolerance in coordinates
2. This matches locations that differ by up to 0.56 meters
3. Preserve original 6-8 decimal place coordinates from both datasets
4. Remove rounded temporary columns after merge
5. Inner join ensures both datasets must have matching date + location


In [13]:
print("Merging datasets with tolerance-based coordinate matching...\n")
print(f"Merge type: Inner join (only matching rows in both datasets)")
print(f"Key fields: date, latitude (rounded to {TOLERANCE_DECIMALS} decimals), longitude (rounded to {TOLERANCE_DECIMALS} decimals)\n")

merged_df = pd.merge(
    gee_match,
    pm25_match,
    on=['date', 'lat_rounded', 'lon_rounded'],
    how='inner',
    suffixes=('_gee', '_pm25')
)

print(f"✓ Merge completed successfully!")
print(f"Merged data shape: {merged_df.shape[0]} rows × {merged_df.shape[1]} columns")

# Now we need to consolidate the coordinate columns
# Use GEE coordinates as primary, but note PM25 coordinates
if 'latitude_gee' in merged_df.columns:
    merged_df['latitude'] = merged_df['latitude_gee']
    merged_df['longitude'] = merged_df['longitude_gee']
    print(f"\nNote: Using GEE coordinates (latitude_gee, longitude_gee) as primary")
    print(f"      PM25 coordinates (latitude_pm25, longitude_pm25) are also available")

Merging datasets with tolerance-based coordinate matching...

Merge type: Inner join (only matching rows in both datasets)
Key fields: date, latitude (rounded to 5 decimals), longitude (rounded to 5 decimals)

✓ Merge completed successfully!
Merged data shape: 41718 rows × 19 columns

Note: Using GEE coordinates (latitude_gee, longitude_gee) as primary
      PM25 coordinates (latitude_pm25, longitude_pm25) are also available


## Step 7: Clean Up Temporary Columns


In [14]:
print("Cleaning up temporary rounded coordinate columns...\n")

# Show columns before cleanup
print(f"Columns before cleanup ({len(merged_df.columns)}):") 
print(f"  {list(merged_df.columns)}\n")

# Drop the temporary rounded columns
merged_df = merged_df.drop(['lat_rounded', 'lon_rounded'], axis=1)

print(f"✓ Removed temporary rounded coordinate columns")
print(f"\nColumns after cleanup ({len(merged_df.columns)}):") 
for i, col in enumerate(merged_df.columns, 1):
    print(f"  {i:2d}. {col}")

Cleaning up temporary rounded coordinate columns...

Columns before cleanup (21):
  ['date', 'datetime_utc', 'id', 'latitude_gee', 'longitude_gee', 'temperature_celsius', 'pressure_mb', 'wind_u', 'wind_v', 'NO2', 'CO', 'O3', 'AOD', 'lat_rounded', 'lon_rounded', 'latitude_pm25', 'longitude_pm25', 'pm25', 'sensor_name', 'latitude', 'longitude']

✓ Removed temporary rounded coordinate columns

Columns after cleanup (19):
   1. date
   2. datetime_utc
   3. id
   4. latitude_gee
   5. longitude_gee
   6. temperature_celsius
   7. pressure_mb
   8. wind_u
   9. wind_v
  10. NO2
  11. CO
  12. O3
  13. AOD
  14. latitude_pm25
  15. longitude_pm25
  16. pm25
  17. sensor_name
  18. latitude
  19. longitude


## Step 8: Sort Data for Readability


In [15]:
print("Sorting data by date and coordinates...\n")

# Use the primary coordinates (GEE) for sorting
if 'latitude' in merged_df.columns:
    merged_df = merged_df.sort_values(['date', 'latitude', 'longitude']).reset_index(drop=True)
else:
    merged_df = merged_df.sort_values(['date', 'latitude_gee', 'longitude_gee']).reset_index(drop=True)

print("✓ Data sorted by date, latitude, longitude")
print(f"\nFirst few rows of merged data:")
print(merged_df.head())

Sorting data by date and coordinates...

✓ Data sorted by date, latitude, longitude

First few rows of merged data:
        date         datetime_utc  id  latitude_gee  longitude_gee  \
0 2016-11-25  2016-11-25 00:00:00   1     38.788056      -0.697222   
1 2016-11-25  2016-11-25 00:00:00  36     39.945278      -0.056389   
2 2016-11-30  2016-11-30 00:00:00  35     40.051944      -0.189722   
3 2016-11-30  2016-11-30 00:00:00  37     40.062222       0.072778   
4 2016-12-02  2016-12-02 00:00:00   1     38.788056      -0.697222   

   temperature_celsius  pressure_mb    wind_u    wind_v  NO2  CO  O3     AOD  \
0             6.493180   946.288660  0.230240  0.735303  NaN NaN NaN  0.1110   
1             9.251236   993.626994  0.783218 -0.592618  NaN NaN NaN  0.0220   
2            10.044782   981.911432 -2.211047 -1.568971  NaN NaN NaN  0.2380   
3            12.299584  1008.794766 -3.236886 -2.804465  NaN NaN NaN  0.2480   
4             9.350630   953.114180  0.953454  0.293060  NaN Na

## Step 9: Inspect Coordinate Differences (Optional Quality Check)

In [16]:
print("Coordinate difference analysis (GEE vs PM25 on matched rows):\n")

if 'latitude_gee' in merged_df.columns and 'latitude_pm25' in merged_df.columns:
    lat_diff = abs(merged_df['latitude_gee'] - merged_df['latitude_pm25'])
    lon_diff = abs(merged_df['longitude_gee'] - merged_df['longitude_pm25'])
    
    print(f"Latitude differences (GEE vs PM25):")
    print(f"  Min:    {lat_diff.min():.8f}°")
    print(f"  Max:    {lat_diff.max():.8f}°")
    print(f"  Mean:   {lat_diff.mean():.8f}°")
    print(f"  Median: {lat_diff.median():.8f}°")
    
    print(f"\nLongitude differences (GEE vs PM25):")
    print(f"  Min:    {lon_diff.min():.8f}°")
    print(f"  Max:    {lon_diff.max():.8f}°")
    print(f"  Mean:   {lon_diff.mean():.8f}°")
    print(f"  Median: {lon_diff.median():.8f}°")
    
    print(f"\nAll differences within tolerance? {(lat_diff.max() <= 0.00001) and (lon_diff.max() <= 0.00001)}")
else:
    print("(Coordinate columns from both sources not available for comparison)")

Coordinate difference analysis (GEE vs PM25 on matched rows):

Latitude differences (GEE vs PM25):
  Min:    0.00000000°
  Max:    0.00000000°
  Mean:   0.00000000°
  Median: 0.00000000°

Longitude differences (GEE vs PM25):
  Min:    0.00000000°
  Max:    0.00000000°
  Mean:   0.00000000°
  Median: 0.00000000°

All differences within tolerance? True


## Step 10: Save Combined Dataset

In [17]:
print("Saving combined dataset...\n")

output_path = data_dir / 'combined_gee_pm25_v2.csv'
merged_df.to_csv(output_path, index=False)

print(f"✓ Output saved to: {output_path}")
print(f"\nFile size: {output_path.stat().st_size / 1024:.2f} KB")

Saving combined dataset...

✓ Output saved to: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\data\combined_gee_pm25_v2.csv

File size: 12516.07 KB


## Summary and Verification

In [18]:
print("="*70)
print("MERGE SUMMARY (V2 - TOLERANCE-BASED MATCHING)")
print("="*70)

print(f"\nMatching Configuration:")
print(f"  Tolerance (decimal places): {TOLERANCE_DECIMALS}")
print(f"  Tolerance (meters):         ~1.1 meters")
print(f"  Coordinate precision:       ~0.56 meters")

print(f"\nInput files:")
print(f"  PM25:                {pm25_df.shape[0]:,} rows")
print(f"  GEE Features:        {gee_df.shape[0]:,} rows")

print(f"\nOutput file (combined_gee_pm25_v2.csv):")
print(f"  Rows:                {merged_df.shape[0]:,}")
print(f"  Columns:             {merged_df.shape[1]}")

print(f"\nData coverage:")
if 'latitude' in merged_df.columns:
    print(f"  Date range:          {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
    print(f"  Unique dates:        {merged_df['date'].nunique()}")
    print(f"  Unique locations:    {len(merged_df[['latitude', 'longitude']].drop_duplicates())}")
else:
    print(f"  Date range:          {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
    print(f"  Unique dates:        {merged_df['date'].nunique()}")
    print(f"  Unique locations:    {len(merged_df[['latitude_gee', 'longitude_gee']].drop_duplicates())}")

print(f"\nMissing values per column:")
missing = merged_df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  ✓ No missing values!")
else:
    for col, count in missing_cols.items():
        pct = (count / len(merged_df)) * 100
        print(f"  {col}: {count:,} ({pct:.1f}%)")

print("\n" + "="*70)
print("✓ Merge completed successfully!")
print("="*70)

MERGE SUMMARY (V2 - TOLERANCE-BASED MATCHING)

Matching Configuration:
  Tolerance (decimal places): 5
  Tolerance (meters):         ~1.1 meters
  Coordinate precision:       ~0.56 meters

Input files:
  PM25:                88,041 rows
  GEE Features:        164,208 rows

Output file (combined_gee_pm25_v2.csv):
  Rows:                41,718
  Columns:             19

Data coverage:
  Date range:          2016-11-25 to 2026-04-06
  Unique dates:        2305
  Unique locations:    98

Missing values per column:
  NO2: 2,410 (5.8%)
  CO: 2,410 (5.8%)
  O3: 2,451 (5.9%)

✓ Merge completed successfully!
